In [2]:
from shapely import wkt
from shapely.ops import transform
import geopandas as gpd
import numpy as np
import pandas as pd

In [3]:
clients_with_coords = pd.read_csv("../output/clients.csv")
employees_with_coords = pd.read_csv("../output/employees.csv")
heerlen_edge_table = pd.read_csv("../output/heerlen_edge_table.csv")

In [4]:
def parse_coordinates(value):
    try:
        first_part, second_part = str(value).strip().split()
        latitude = float(first_part)
        longitude = float(second_part)
        return latitude, longitude
    except Exception:
        return None


def parse_edge_geometry(value):
    try:
        return wkt.loads(str(value))
    except Exception:
        return None


def swap_xy(geometry):
    return transform(lambda x, y, z=None: (y, x), geometry)


def geometry_looks_reversed(geometry):
    if geometry is None or geometry.is_empty:
        return False
    if hasattr(geometry, "geoms"):
        coords = [coord for part in geometry.geoms for coord in part.coords]
    elif hasattr(geometry, "coords"):
        coords = list(geometry.coords)
    else:
        return False

    if not coords:
        return False

    x_values = [coord[0] for coord in coords]
    y_values = [coord[1] for coord in coords]
    return float(np.median(x_values)) > float(np.median(y_values))


# Build client points from the existing "coordinates" column, formatted as "latitude longitude".
clients_df = clients_with_coords.copy()
clients_df[["latitude", "longitude"]] = clients_df["coordinates"].apply(
    lambda value: pd.Series(parse_coordinates(value))
)
clients_df = clients_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
clients_gdf = gpd.GeoDataFrame(
    clients_df,
    geometry=gpd.points_from_xy(clients_df["longitude"], clients_df["latitude"]),
    crs="EPSG:4326",
)

# Build employee points from the existing "coordinates" column.
employees_df = employees_with_coords.copy()
employees_df[["latitude", "longitude"]] = employees_df["coordinates"].apply(
    lambda value: pd.Series(parse_coordinates(value))
)
employees_df = employees_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
employees_gdf = gpd.GeoDataFrame(
    employees_df,
    geometry=gpd.points_from_xy(employees_df["longitude"], employees_df["latitude"]),
    crs="EPSG:4326",
)

# Normalize edge geometries if they were stored as "latitude longitude" instead of "longitude latitude".
edges_df = heerlen_edge_table.copy()
edges_df["geometry"] = edges_df["geometry"].apply(parse_edge_geometry)
edges_df = edges_df.dropna(subset=["geometry"]).reset_index(drop=True)
edges_df["edge_id"] = edges_df.index
edges_df["geometry"] = edges_df["geometry"].apply(
    lambda geometry: swap_xy(geometry) if geometry_looks_reversed(geometry) else geometry
)
edges_gdf = gpd.GeoDataFrame(edges_df, geometry="geometry", crs="EPSG:4326")

# Project to a metric CRS before measuring distances.
projected_crs = clients_gdf.estimate_utm_crs()
if projected_crs is None:
    projected_crs = employees_gdf.estimate_utm_crs()
if projected_crs is None:
    projected_crs = "EPSG:3857"

clients_projected = clients_gdf.to_crs(projected_crs)
employees_projected = employees_gdf.to_crs(projected_crs)
edges_projected = edges_gdf.to_crs(projected_crs)

In [5]:
# Find the nearest edge segment for each client.
nearest_clients = gpd.sjoin_nearest(
    clients_projected,
    edges_projected[["edge_id", "u", "v", "key", "name", "highway", "length", "travel_time", "travel_time_min", "geometry"]],
    how="left",
    distance_col="distance_m",
)

# GeoPandas can return multiple rows for equally-near edges; keep one match per client.
nearest_clients = (
    nearest_clients.reset_index(names="source_index")
    .sort_values(["source_index", "distance_m", "edge_id"], kind="stable")
    .drop_duplicates(subset=["source_index"], keep="first")
)

client_nearest_segments = nearest_clients[[
    "name_left",
    "address",
    "coordinates",
    "edge_id",
    "u",
    "v",
    "key",
    "name_right",
    "highway",
    "distance_m",
]].rename(
    columns={
        "name_left": "client_name",
        "name_right": "edge_name",
    }
)

client_nearest_segments.to_csv("../output/client_nearest_segments.csv", index=False)

# Find the nearest edge segment for each employee.
nearest_employees = gpd.sjoin_nearest(
    employees_projected,
    edges_projected[["edge_id", "u", "v", "key", "name", "highway", "length", "travel_time", "travel_time_min", "geometry"]],
    how="left",
    distance_col="distance_m",
)

# GeoPandas can return multiple rows for equally-near edges; keep one match per employee.
nearest_employees = (
    nearest_employees.reset_index(names="source_index")
    .sort_values(["source_index", "distance_m", "edge_id"], kind="stable")
    .drop_duplicates(subset=["source_index"], keep="first")
)

employee_nearest_segments = nearest_employees[[
    "name_left",
    "address",
    "coordinates",
    "edge_id",
    "u",
    "v",
    "key",
    "name_right",
    "highway",
    "distance_m",
]].rename(
    columns={
        "name_left": "employee_name",
        "name_right": "edge_name",
    }
)

employee_nearest_segments.to_csv("../output/employee_nearest_segments.csv", index=False)